# AHC015 T4 x2 speed benchmark

GPU T4 x2とInternetを有効にし、`GITHUB_TOKEN`と`WANDB_API_KEY`へのSecret accessを許可して実行する。学習checkpointは変更しない。

In [ ]:
import torch

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
for index in range(2):
    print(index, torch.cuda.get_device_name(index), torch.cuda.get_device_capability(index))
    assert torch.cuda.get_device_capability(index) == (7, 5)

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repo_dir = Path("/kaggle/working/ahc-ml")
expected_commit = "8f678d9b90258191c314e611078bc8a6729375d0"
assert not repo_dir.exists()
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credentials}"
try:
    subprocess.run(
        [
            "git", "clone", "--branch", "feature/ahc015-teacher",
            "--single-branch", "https://github.com/e1jirou/ahc-ml.git", str(repo_dir),
        ],
        check=True,
        env=git_env,
    )
finally:
    del github_token, credentials, git_env
subprocess.run(
    ["git", "-C", str(repo_dir), "checkout", "--detach", expected_commit],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == expected_commit
print("Repository commit:", actual_commit)

In [ ]:
import os
from pathlib import Path

import wandb
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
checkpoint_dir = Path("/kaggle/working/checkpoints/ppo-20260826-000916")
artifact = wandb.Api().artifact(
    "eijirou-personal/ahc-ml/ppo-20260826-000916-training-checkpoint:latest",
    type="model",
)
downloaded_dir = Path(artifact.download(root=checkpoint_dir))
checkpoint_path = downloaded_dir / "best-training.pt"
assert checkpoint_path.is_file()
print("Checkpoint:", checkpoint_path)

In [ ]:
import os
import subprocess
import sys

benchmark_env = os.environ.copy()
benchmark_env["PYTHONPATH"] = str(repo_dir / "python")
benchmark_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
base_arguments = [
    sys.executable,
    "-m",
    "examples.ahc015.python.benchmark_training",
    "--checkpoint",
    str(checkpoint_path),
    "--device",
    "cuda",
    "--episodes",
    "1024",
    "--batch-size",
    "1024",
    "--update-batches",
    "4",
]
configurations = [
    ("single-128", ["--micro-batch-size", "128"]),
    ("single-256", ["--micro-batch-size", "256"]),
    ("single-512", ["--micro-batch-size", "512"]),
    ("dual-256", ["--micro-batch-size", "256", "--data-parallel"]),
    ("dual-512", ["--micro-batch-size", "512", "--data-parallel"]),
    ("dual-1024", ["--micro-batch-size", "1024", "--data-parallel"]),
]

for name, extra_arguments in configurations:
    print(f"\n===== {name} =====", flush=True)
    result = subprocess.run(
        base_arguments + extra_arguments,
        cwd=repo_dir,
        env=benchmark_env,
        check=False,
    )
    print(f"{name} return code: {result.returncode}", flush=True)